# 02 — Fraude bancária (Credit Card Fraud Detection, ULB)

Classe positiva: **é fraude?** (`target = 1`). Execução pesada: rodar no Colab. Checkpoints em `outputs/fraude/models/`.

In [ ]:
# Configuração do ambiente (Google Colab). Fora do Colab, usa a pasta local do projeto.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/a2'   # fácil de trocar
    IN_COLAB = True
except ImportError:
    import os
    PROJECT_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    IN_COLAB = False
%cd {PROJECT_DIR}
if IN_COLAB:
    !pip install -q -r requirements.txt
import sys; sys.path.insert(0, PROJECT_DIR)

In [ ]:
DOMAIN = "fraude"
FORCE_RETRAIN = False   # True = ignora os checkpoints em outputs/fraude/models/ e retreina

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src import config, cutoff_hist, evaluate, explain, pipeline
from src.pipeline import timed

cfg = config.DOMAINS[DOMAIN]
OUT = config.output_dir(DOMAIN)
FX = OUT / "faixas"
T = config.THRESHOLD
FIGS = []

def show(path):
    """Registra a figura para o resumo e exibe no notebook."""
    FIGS.append(path.relative_to(OUT).as_posix())
    display(Image(filename=str(path)))

def table(df, stem):
    """Salva CSV + Markdown em outputs/<dominio>/ e exibe."""
    evaluate.save_table(df, OUT / stem)
    display(evaluate.round_table(df))

print(f"Domínio: {DOMAIN} | classe positiva: {cfg['positive_question']} | "
      f"métrica-alvo: {cfg['target_metric']} | limiar das métricas: {T}")

## 1. Carregar os dados

In [ ]:
with timed("carregar dados"):
    X, y = pipeline.load_domain_data(DOMAIN)
print(f"X: {X.shape} | positivos (target = 1): {y.sum()} ({100 * y.mean():.2f}%)")

In [ ]:
print(X.columns.tolist())
assert {"Time", "Amount"} <= set(X.columns)

## 2. Balanceamento das classes

In [ ]:
balanceamento = evaluate.class_balance_table(y, cfg)
table(balanceamento, "balanceamento")
show(evaluate.plot_class_balance(y, cfg, OUT / "01_balanceamento_classes.png",
                                 f"Balanceamento das classes — {DOMAIN} (N = {len(y)})"))

**Atenção:** desbalanceamento extremo (~0,17% de fraude). Um modelo que responde sempre "legítima" teria ~99,8% de acurácia, por isso a acurácia é enganosa neste domínio. No histograma da Parte 1, as barras vermelhas ficam muito pequenas porque o denominador é N (comportamento exigido).

## 3. Split estratificado 60/20/20 (treino / validação / teste)

In [ ]:
with timed("split"):
    splits = pipeline.split_data(X, y)
X_tr, y_tr = splits["treino"]
X_val, y_val = splits["validacao"]
X_te, y_te = splits["teste"]
table(pipeline.split_table(splits), "split")

## 4. Tuning por validação cruzada (somente no treino)

Pipeline completo (pré-processamento + modelo) dentro da busca; `StratifiedKFold(n_splits=5, shuffle=True)`. Checkpoint em `models/`.

In [ ]:
tuned = pipeline.tune_all(DOMAIN, X_tr, y_tr, force_retrain=FORCE_RETRAIN)
hiperparametros = pipeline.hyperparams_table(tuned, cfg["target_metric"])
table(hiperparametros, "hiperparametros")

## 5. Comparação dos modelos na validação (limiar = 0,5 nas métricas de decisão)

In [ ]:
with timed("avaliação na validação"):
    val_probas = {name: pipeline.predict_positive(ck["estimator"], X_val)
                  for name, ck in tuned.items()}
    comparacao = evaluate.comparison_table(y_val, val_probas)
table(comparacao, "comparacao_validacao")
for name, p in val_probas.items():
    show(evaluate.plot_confusion(
        y_val, p, cfg, OUT / f"02_matriz_confusao_validacao_{pipeline.slug(name)}.png",
        f"Matriz de confusão — {name} — validação"))

## 6. Seleção do melhor modelo (métrica-alvo na validação)

In [ ]:
best_name, best_col, best_value = evaluate.select_best(comparacao, cfg["target_metric"])
best = tuned[best_name]["estimator"]
selecao = f"{best_name} — maior {best_col} na validação = {best_value:.4f}"
print("Melhor modelo:", selecao)

## 7. Curvas ROC e PR na validação

In [ ]:
n_val = len(y_val)
show(evaluate.plot_roc(y_val, val_probas, OUT / "03_curva_roc_validacao.png",
                       f"Curva ROC — {DOMAIN} — validação (N = {n_val})"))
show(evaluate.plot_pr(y_val, val_probas, OUT / "04_curva_pr_validacao.png",
                      f"Curva Precisão-Revocação — {DOMAIN} — validação (N = {n_val})"))

## 8. Varredura de limiar do melhor modelo (validação)

In [ ]:
varredura = evaluate.threshold_sweep(y_val, val_probas[best_name])
table(varredura, "varredura_limiar")

## 9. Avaliação final no teste (melhor modelo, sem retreinar e sem ajustes)

In [ ]:
with timed("avaliação no teste"):
    p_test = pipeline.predict_positive(best, X_te)
    resultado_teste = evaluate.comparison_table(y_te, {best_name: p_test})
resultado_teste.insert(1, "conjunto", "teste")
table(resultado_teste, "resultado_teste")
n_te = len(y_te)
show(evaluate.plot_confusion(y_te, p_test, cfg, OUT / "05_matriz_confusao_teste.png",
                             f"Matriz de confusão — {best_name} — teste"))
show(evaluate.plot_roc(y_te, {best_name: p_test}, OUT / "06_curva_roc_teste.png",
                       f"Curva ROC — {DOMAIN} — teste (N = {n_te})"))
show(evaluate.plot_pr(y_te, {best_name: p_test}, OUT / "07_curva_pr_teste.png",
                      f"Curva Precisão-Revocação — {DOMAIN} — teste (N = {n_te})"))

## 10. SHAP beeswarm do melhor modelo (amostra do teste)

In [ ]:
with timed("SHAP"):
    shap_res = explain.explain_model(DOMAIN, best_name, best, X_tr, X_te, OUT,
                                     OUT / "08_shap_beeswarm.png")
show(OUT / "08_shap_beeswarm.png")
print("\n".join(shap_res["info"]))
display(evaluate.round_table(shap_res["top10"]))

## 11. Parte 1 aplicada ao melhor modelo

Cortes escolhidos **somente na validação** e aplicados ao teste **sem ajuste**.

In [ ]:
p_val = val_probas[best_name]
(t1, t2), rep_val, tradeoff = cutoff_hist.choose_cutoffs(
    y_val, p_val, cfg["max_fn_rate"], cfg["max_fp_rate"])
print(f"t1 = {t1:.2f} | t2 = {t2:.2f}  (max_fn_rate = {cfg['max_fn_rate']} sobre positivos, "
      f"max_fp_rate = {cfg['max_fp_rate']} sobre negativos)")
table(tradeoff, "faixas/tradeoff_validacao")
table(rep_val["faixas"], "faixas/band_report_validacao_faixas")
table(rep_val["erros"], "faixas/band_report_validacao_erros")
fig, _ = cutoff_hist.plot_cutoff_histogram(
    y_val, p_val, cfg["bin_width"], t1, t2, cfg["positive_label"],
    f"Faixas de decisão — {DOMAIN} — {best_name}", FX / "09_histograma_faixas_validacao.png",
    dataset_name="validação")
plt.close(fig)
show(FX / "09_histograma_faixas_validacao.png")

In [ ]:
rep_test = cutoff_hist.band_report(y_te, p_test, t1, t2)   # mesmos t1 e t2 da validação
table(rep_test["faixas"], "faixas/band_report_teste_faixas")
table(rep_test["erros"], "faixas/band_report_teste_erros")
fig, _ = cutoff_hist.plot_cutoff_histogram(
    y_te, p_test, cfg["bin_width"], t1, t2, cfg["positive_label"],
    f"Faixas de decisão — {DOMAIN} — {best_name}", FX / "10_histograma_faixas_teste.png",
    dataset_name="teste")
plt.close(fig)
show(FX / "10_histograma_faixas_teste.png")
pd.DataFrame({"y_true": y_val, "y_proba": p_val}).to_csv(OUT / "preds_validacao.csv", index=False)
pd.DataFrame({"y_true": y_te, "y_proba": p_test}).to_csv(OUT / "preds_teste.csv", index=False)

## 12. Resumo numérico

In [ ]:
tempos = pd.DataFrame({"etapa": list(pipeline.TIMINGS), "segundos": list(pipeline.TIMINGS.values())})
blocos = [
    ("Balanceamento", balanceamento),
    ("Split 60/20/20", pipeline.split_table(splits)),
    (f"Hiperparâmetros vencedores (CV {config.CV_SPLITS} folds, {cfg['target_metric']})",
     hiperparametros),
    (f"Comparação na validação (limiar = {T})", comparacao),
    ("Melhor modelo", selecao),
    (f"Resultado no teste (limiar = {T})", resultado_teste),
    ("SHAP — informações", shap_res["info"]),
    ("SHAP — top 10 features (média |SHAP|)", shap_res["top10"]),
    ("Cortes (escolhidos na validação)",
     [f"t1 = {t1:.2f}", f"t2 = {t2:.2f}",
      f"max_fn_rate = {cfg['max_fn_rate']} (sobre positivos)",
      f"max_fp_rate = {cfg['max_fp_rate']} (sobre negativos)"]),
    ("Trade-off (10 melhores pares viáveis, validação)", tradeoff),
    ("band_report — validação — faixas", rep_val["faixas"]),
    ("band_report — validação — erros", rep_val["erros"].drop(columns="descricao")),
    ("band_report — teste — faixas", rep_test["faixas"]),
    ("band_report — teste — erros", rep_test["erros"].drop(columns="descricao")),
    ("Tempo de execução (s)", tempos),
    ("Figuras geradas", FIGS),
]
resumo = evaluate.write_summary(OUT / "resumo_resultados.md",
                                f"Resumo de resultados — {DOMAIN}", blocos)
print("Resumo salvo em", resumo)